In [1]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Import module from src directory
from preprocessing_mask_segmentation import preprocessing as pre
from shape_features import shape_features as sf
from pyrandomics_playground import pyrandomics_implementation as pr
from split import split
from feature_engineering import selection as sel
from resize_image import resize as res

In [2]:
# Constant variables
FOLDER_PATH = "../images/"
CSV_PATH = "../labels.csv"
MERGED_CSV_AND_FEATURE_PATH = "../features.csv"

In [3]:
images_names = sorted([f for f in os.listdir(FOLDER_PATH) if f.endswith('.png')])

# Import and preprocess images
images_dict = {}
for image_name in images_names:
    full_path = os.path.join(FOLDER_PATH, image_name)
    key = image_name.split('.')[0]
    img = pre.clean_mask(full_path)

    # Image Resizing - uncomment if CNN used
    # img_resized = res.resize_with_padding(img)
    images_dict[key] = img

# Extract features from images
features_results = []

for name, img in images_dict.items():
    full_path = os.path.join(FOLDER_PATH, name+".png")
    features = sf.extract_shape_features(img) # Extract features using region props based module
    pr_features = pr.extract_shape_features_from_mask(full_path) # Extract features using pyradiomics based module
    features.update(pr_features) # Update features dictionary with pyradiomics features
    if features:
        features['id'] = name
        features_results.append(features)
    else:
        print(f"{name} omitted (empty mask or unexpected error).")

In [4]:
# Import CSV file and
csv_file = pd.read_csv(CSV_PATH)

# Convert features_results into Pandas DataFrame
df_feature_results = pd.DataFrame(features_results)

# Convert id columns to string to prepare merge
csv_file['id'] = csv_file['id'].astype(str)
df_feature_results["id"] = df_feature_results["id"].astype(str)

# Merge CSV columns and feature DataFrame into one final DataFrame
df_final = pd.merge(csv_file, df_feature_results, left_on='id', right_on='id')

# Drop useless columns and convert target feature into binary value
df_final.drop(columns='abnormality_type', inplace=True)
mapping = {'MALIGNANT': 1, 'BENIGN': 0}
df_final['pathology'] = df_final['pathology'].map(mapping)

# Export final DataFrame into CSV file
df_final.to_csv(MERGED_CSV_AND_FEATURE_PATH, index=False)

## Test of Feature Extraction and Preprocessing

In [5]:
df_final.head()

,id,patient_id,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,...,Hu Moment 7,original_shape2D_Elongation,original_shape2D_MajorAxisLength,original_shape2D_MaximumDiameter,original_shape2D_MeshSurface,original_shape2D_MinorAxisLength,original_shape2D_Perimeter,original_shape2D_PerimeterSurfaceRatio,original_shape2D_PixelSurface,original_shape2D_Sphericity
0,1001,P_00001,4,1,120176.0,184775.0,144592.0,120176.0,437.817532,369.842129,...,41.60581,0.840962,447.005783,502.236996,124054.0,375.915034,11341.453814,0.091424,124881.0,0.110088
1,1002,P_00001,4,1,32114.0,54432.0,39160.0,32114.0,215.156997,204.049262,...,35.15640,0.943530,221.291459,262.937255,33513.0,208.795081,4451.100356,0.132817,33875.0,0.145796
2,1003,P_00004,4,0,113057.0,162306.0,132259.0,113057.0,403.087349,373.343420,...,-39.57749,0.925515,409.909988,452.659916,116337.0,379.377683,10081.557277,0.086658,117128.0,0.119932
3,1004,P_00004,4,0,89182.0,147452.0,111392.0,89182.0,361.346738,333.637192,...,-39.40404,0.923041,369.899218,429.382114,92938.0,341.432140,11268.303669,0.121245,93806.0,0.095905
4,1005,P_00004,4,0,91546.0,151434.0,108858.0,91546.0,389.640992,311.266317,...,38.29029,0.801773,396.665021,441.897047,94536.0,318.035325,9069.472791,0.095937,95248.0,0.120177


In [6]:
df_final.describe()

,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,Centroid X,Centroid Y,...,Hu Moment 7,original_shape2D_Elongation,original_shape2D_MajorAxisLength,original_shape2D_MaximumDiameter,original_shape2D_MeshSurface,original_shape2D_MinorAxisLength,original_shape2D_Perimeter,original_shape2D_PerimeterSurfaceRatio,original_shape2D_PixelSurface,original_shape2D_Sphericity
count,1664.000000,1664.000000,1.664000e+03,1.664000e+03,1.664000e+03,1.664000e+03,1664.000000,1664.000000,1664.000000,1664.000000,...,1664.000000,1664.000000,1664.000000,1664.000000,1.664000e+03,1664.000000,1664.000000,1664.000000,1.664000e+03,1664.000000
mean,3.501202,0.458534,7.868357e+04,1.221009e+05,9.073593e+04,7.868357e+04,334.169193,269.915583,206.031864,202.283244,...,0.093411,0.815981,339.966279,373.587871,8.088757e+04,275.892554,6771.636978,0.109066,8.141759e+04,0.144375
std,1.404828,0.498427,8.055054e+04,1.206283e+05,9.094821e+04,8.055054e+04,137.835099,113.287892,83.124890,81.275151,...,37.016106,0.105319,137.912696,146.274700,8.145605e+04,113.350225,3408.396570,0.040458,8.167956e+04,0.030806
min,0.000000,0.000000,3.966000e+03,6.095000e+03,4.643000e+03,3.966000e+03,80.596272,52.545921,51.516357,53.855270,...,-47.280762,0.235148,85.949642,99.764723,4.462000e+03,56.021813,1176.302741,0.012086,4.588000e+03,0.078946
25%,3.000000,0.000000,3.621375e+04,5.792400e+04,4.223800e+04,3.621375e+04,245.383208,195.395222,152.343274,148.652792,...,-36.714913,0.759485,251.109973,279.028672,3.776600e+04,201.410715,4662.178567,0.082740,3.813025e+04,0.126931
50%,4.000000,0.000000,5.651000e+04,8.885800e+04,6.547500e+04,5.651000e+04,305.682136,247.529924,188.373131,186.009010,...,26.018202,0.832634,311.029433,342.118299,5.848175e+04,253.375901,6003.902664,0.103361,5.902050e+04,0.140712
75%,4.000000,1.000000,9.042775e+04,1.425425e+05,1.045720e+05,9.042775e+04,383.560436,318.783599,237.281471,236.437932,...,36.752786,0.892520,389.353409,428.077388,9.278538e+04,325.452030,7872.580692,0.130612,9.339600e+04,0.155642
max,5.000000,1.000000,1.251936e+06,1.797881e+06,1.345671e+06,1.251936e+06,1317.697863,1222.367729,811.905826,791.568087,...,49.230771,0.991157,1325.946937,1380.488682,1.266556e+06,1230.996844,35311.304151,0.342320,1.269045e+06,0.469933


## Select features

In [17]:
metadata_cols = ['id', 'patient_id', 'pathology', 'assessment']
df_dropped_cols = sel.drop_high_correlation_features(df_final, exclude=metadata_cols)

df_cleaned = df_dropped_cols[0]
dropped_features = df_dropped_cols[1]

# Drop remaining scale dependent features
remaining_scale_dependent_features = ["Area", "Orientation"]
df_cleaned = df_cleaned.drop(columns=remaining_scale_dependent_features)

Dropped 28 redundant features due to high correlation (> 0.9): ['Area Bounding Box', 'Area Convex', 'Area Filled', 'Axis Major Length', 'Axis Minor Length', 'Centroid X', 'Centroid Y', 'Equivalent Diameter', 'Feret Diameter Max', 'Perimeter', 'Aspect Ratio', 'Convexity', 'Irregularity Index', 'Roundness', 'Shape Factor', 'Relative Area', 'Hu Moment 1', 'Hu Moment 2', 'Hu Moment 3', 'original_shape2D_Elongation', 'original_shape2D_MajorAxisLength', 'original_shape2D_MaximumDiameter', 'original_shape2D_MeshSurface', 'original_shape2D_MinorAxisLength', 'original_shape2D_Perimeter', 'original_shape2D_PerimeterSurfaceRatio', 'original_shape2D_PixelSurface', 'original_shape2D_Sphericity']


## Data Split
Perform `patient_id` aware split. Extract X features and y labels into separate DataFrames. Split X and y into train and test sets.

In [18]:
drop_cols = ['id', 'patient_id', 'pathology', 'assessment']

X_train, X_test, y_train, y_test, is_intersect_empty = split.aware_patient_split(df_cleaned, drop_cols)

# Verify that the intersection of patient IDs in Train and Test sets is empty
is_intersect_empty  # Result: True

True

## Scale Features with StandardScaler

In [19]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [20]:
X_train_scaled


,Eccentricity,Extent,Solidity,Circularity,Hu Moment 4,Hu Moment 5,Hu Moment 6,Hu Moment 7
0,-0.124308,0.187877,-0.621119,-1.258469,-1.475013,-0.980892,-0.984830,1.122194
1,-1.601743,-0.760369,-0.838408,-0.335424,0.320042,-0.807333,-0.806241,0.947801
2,-1.196140,0.912727,-0.156261,-1.103996,-0.757207,-0.935025,-0.905031,-1.073016
3,-1.148483,-0.527469,-1.220431,-1.655467,-0.807764,-0.929317,-0.915499,-1.068325
4,0.325305,-0.532071,-0.428150,-0.898501,-0.292073,1.190408,1.204113,1.032542
...,...,...,...,...,...,...,...,...
1656,-2.490303,0.751277,0.877563,0.224618,-0.405472,1.181105,-0.834769,0.987422
1657,-0.259660,-0.352950,0.332652,0.126015,0.075716,-0.830386,-0.831414,-1.013446
1661,2.313035,-1.169590,-3.081416,-1.075267,-0.172415,-0.866480,-0.883015,0.982572
1662,-0.176198,0.168585,0.051762,-0.769851,-0.449023,-0.880970,-0.908666,-1.051543


## Model Training

Basic model training - for test purposes.

In [21]:
from models import models

lr = models.train_lr(X_train_scaled, y_train)
svm = models.train_svm(X_train_scaled, y_train)
rf = models.train_rf(X_train_scaled, y_train)
knn = models.train_knn(X_train_scaled, y_train)

## Model Tests

In [22]:
result_lr = models.test_model(lr, X_test_scaled, y_test)
result_svm = models.test_model(svm, X_test_scaled, y_test)
result_rf = models.test_model(rf, X_test_scaled, y_test)
result_knn = models.test_model(knn, X_test_scaled, y_test)

Model: LogisticRegression(max_iter=2000, random_state=42)

    Accuracy: 0.589
    Recall: 0.414
    Precision: 0.575
    F1-Score: 0.481
    

    Confusion Matrix: [[136  48]
 [ 92  65]]
    

    AUC: 0.577
    
Model: SVC()

    Accuracy: 0.572
    Recall: 0.478
    Precision: 0.540
    F1-Score: 0.507
    

    Confusion Matrix: [[120  64]
 [ 82  75]]
    

    AUC: 0.565
    
Model: RandomForestClassifier()

    Accuracy: 0.545
    Recall: 0.452
    Precision: 0.507
    F1-Score: 0.478
    

    Confusion Matrix: [[115  69]
 [ 86  71]]
    

    AUC: 0.539
    
Model: KNeighborsClassifier()

    Accuracy: 0.507
    Recall: 0.452
    Precision: 0.464
    F1-Score: 0.458
    

    Confusion Matrix: [[102  82]
 [ 86  71]]
    

    AUC: 0.503
    
